In [1]:
import pandas as pd

def get_csv_column_names(file_path):
    df = pd.read_csv(file_path)
    return df.columns.tolist()

# Usage
file_path = 'Almond1.csv'
columns = get_csv_column_names(file_path)
print(columns)

['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Min_Price', 'Max_Price', 'Modal_Price', 'Price_Date']


In [2]:
import pandas as pd
# Assuming your CSV file is named 'data.csv'
df = pd.read_csv('Almond1.csv')
# Get the unique values from the 'District_Name' column
unique_district_names = df['District_Name'].unique()
# Print the unique district names
print(unique_district_names)

['Mewat' 'Shimla' 'Mumbai' 'Angul' 'Cuttack' 'Darjeeling' 'Jashpur'
 'Balrampur' 'Longleng' 'Khammam' 'Aurangabad' 'Rajkot' 'Porbandar'
 'Mahendragarh-Narnaul' 'Bangalore' 'Davangere' 'Mysore' 'Idukki' 'Bhopal'
 'Katni' 'Sagar' 'Ujjain' 'Sangli' 'Ahmednagar' 'Amarawati' 'Nagpur'
 'Chandrapur' 'Jalgaon' 'Pune' 'Nashik' 'Osmanabad' 'Sholapur' 'Kolhapur'
 'Koraput' 'Gajapati' 'Hyderabad' 'Muzaffarnagar' 'Meerut' 'Banaskanth'
 'Amreli' 'Patan' 'Anand' 'Dhar' 'Mandla' 'Satara' 'Kishanganj' 'Buxar'
 'Patna' 'Koria' 'Surat' 'Jammu' 'Panna' 'Bhind' 'Chhindwara' 'Guna'
 'Shehdol' 'Mandsaur' 'Datia' 'Indore' 'Sehore' 'Jhabua' 'Shivpuri'
 'Surguja' 'Dahod' 'Jamnagar' 'Junagarh' 'Gaya' 'Durg' 'Bilaspur'
 'Ahmedabad' 'Madhubani' 'Gariyaband' 'Dhamtari' 'Narayanpur' 'Bharuch'
 'Surendranagar' 'Vadodara(Baroda)' 'Ambala' 'Sirsa' 'Jhajar' 'Fatehabad'
 'Hamirpur' 'Kondagaon' 'Sukma' 'Kanker' 'Hassan' 'Chikmagalur'
 'Karwar(Uttar Kannad)' 'Madikeri(Kodagu)' 'Dindori']


In [3]:
import pandas as pd
# Assuming your CSV file is named 'data.csv'
df = pd.read_csv('Almond1.csv')
# Get the unique values from the 'District_Name' column
unique_district_names = df['Commodity'].unique()
# Print the unique district names
print(unique_district_names)

['Almond(Badam)' 'Bamboo' 'Sabu Dan' 'Seetapal' 'Rajgir' 'Masur Dal'
 'Beans' 'Methi(Leaves)' 'Amla(Nelli Kai)' 'Antawala']


### MIN PRICE (Training)

In [4]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pickle

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('Almond1.csv')

# Extract the features (first 6 labels) and the target variable 'Modal_Price'
X = df[['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Price_Date']]
y = df['Min_Price']

# Feature engineering: Extract month and year from Price_Date
X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
X['Year'] = pd.to_datetime(X['Price_Date']).dt.year
X.drop('Price_Date', axis=1, inplace=True)

# List of categorical columns and numerical columns
categorical_features = ['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Month', 'Year']
numerical_features = ['Month', 'Year']

# Create pipelines for preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Use ColumnTransformer to apply preprocessing to selected categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', 'passthrough', numerical_features)
    ],
    remainder='drop'
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a pipeline with preprocessing and linear regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

# Save the trained model to a file
with open('minprice.pkl', 'wb') as file:
    pickle.dump(model, file)


C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\1391310820.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\1391310820.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\1391310820.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['Year'] = pd.to_datetime(X[

### MIN PRICE (Testing)

In [15]:
import pandas as pd
import pickle

# Load the input data for prediction
input_row = {
    'District_Name': 'Chikmagalur',
    'Market_Name': 'Chikkamagalore',
    'Commodity': 'Antawala',
    'Variety': 'Antawala',
    'Grade': 'FAQ',
    'Price_Date': '9/6/2025'
}

# Load the saved model
with open('minprice.pkl', 'rb') as file:
    model = pickle.load(file)

# Create a DataFrame with the input row
input_df = pd.DataFrame([input_row])

# Feature engineering: Extract month and year from Price_Date
input_df['Month'] = pd.to_datetime(input_df['Price_Date']).dt.month
input_df['Year'] = pd.to_datetime(input_df['Price_Date']).dt.year
input_df.drop('Price_Date', axis=1, inplace=True)

# Make prediction using the loaded model
prediction = model.predict(input_df)
print("Predicted Min Price:", prediction[0])


Predicted Min Price: 5580.635041427449


### MAX PRICE (Training)

In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pickle

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('Almond1.csv')

# Extract the features (first 6 labels) and the target variable 'Modal_Price'
X = df[['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Price_Date']]
y = df['Max_Price']

# Feature engineering: Extract month and year from Price_Date
X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
X['Year'] = pd.to_datetime(X['Price_Date']).dt.year
X.drop('Price_Date', axis=1, inplace=True)

# List of categorical columns and numerical columns
categorical_features = ['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Month', 'Year']
numerical_features = ['Month', 'Year']

# Create pipelines for preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Use ColumnTransformer to apply preprocessing to selected categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', 'passthrough', numerical_features)
    ],
    remainder='drop'
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a pipeline with preprocessing and linear regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

# Save the trained model to a file
with open('maxprice.pkl', 'wb') as file:
    pickle.dump(model, file)


C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\4192404969.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\4192404969.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\4192404969.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['Year'] = pd.to_datetime(X[

### MAX PRICE (Testing)

In [13]:
import pandas as pd
import pickle

# Load the input data for prediction
input_row = {
   'District_Name': 'Chikmagalur',
    'Market_Name': 'Chikkamagalore',
    'Commodity': 'Antawala',
    'Variety': 'Antawala',
    'Grade': 'FAQ',
    'Price_Date': '9/6/2025'
}

# Load the saved model
with open('maxprice.pkl', 'rb') as file:
    model = pickle.load(file)

# Create a DataFrame with the input row
input_df = pd.DataFrame([input_row])

# Feature engineering: Extract month and year from Price_Date
input_df['Month'] = pd.to_datetime(input_df['Price_Date']).dt.month
input_df['Year'] = pd.to_datetime(input_df['Price_Date']).dt.year
input_df.drop('Price_Date', axis=1, inplace=True)

# Make prediction using the loaded model
prediction = model.predict(input_df)
print("Predicted Max Price:", prediction[0])

Predicted Max Price: 5862.157046327135


### MODAL PRICE (Training)

In [8]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pickle

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('Almond1.csv')

# Extract the features (first 6 labels) and the target variable 'Modal_Price'
X = df[['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Price_Date']]
y = df['Modal_Price']

# Feature engineering: Extract month and year from Price_Date
X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
X['Year'] = pd.to_datetime(X['Price_Date']).dt.year
X.drop('Price_Date', axis=1, inplace=True)

# List of categorical columns and numerical columns
categorical_features = ['District_Name', 'Market_Name', 'Commodity', 'Variety', 'Grade', 'Month', 'Year']
numerical_features = ['Month', 'Year']

# Create pipelines for preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Use ColumnTransformer to apply preprocessing to selected categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', 'passthrough', numerical_features)
    ],
    remainder='drop'
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a pipeline with preprocessing and linear regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

# Save the trained model to a file
with open('modalprice.pkl', 'wb') as file:
    pickle.dump(model, file)


C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\1958752899.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\1958752899.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Month'] = pd.to_datetime(X['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_9032\1958752899.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['Year'] = pd.to_datetime(X[

### MODAL PRICE (Testing)

In [2]:
import pandas as pd
import pickle

# Load the input data for prediction
input_row = {
    'District_Name': 'Chikmagalur',
    'Market_Name': 'Chikkamagalore',
    'Commodity': 'Antawala',
    'Variety': 'Antawala',
    'Grade': 'FAQ',
    'Price_Date': '23/7/2025'
}

# Load the saved model
with open('modalprice.pkl', 'rb') as file:
    model = pickle.load(file)

# Create a DataFrame with the input row
input_df = pd.DataFrame([input_row])

# Feature engineering: Extract month and year from Price_Date
input_df['Month'] = pd.to_datetime(input_df['Price_Date']).dt.month
input_df['Year'] = pd.to_datetime(input_df['Price_Date']).dt.year
input_df.drop('Price_Date', axis=1, inplace=True)

# Make prediction using the loaded model
prediction = model.predict(input_df)
print("Predicted Modal Price:", prediction[0])

Predicted Modal Price: 5809.6945413223


C:\Users\sujal\AppData\Local\Temp\ipykernel_120\1643321345.py:22: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  input_df['Month'] = pd.to_datetime(input_df['Price_Date']).dt.month
C:\Users\sujal\AppData\Local\Temp\ipykernel_120\1643321345.py:23: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  input_df['Year'] = pd.to_datetime(input_df['Price_Date']).dt.year
